# What you’ll build (mentally clear picture)

You’ll build a stateful chat agent where:

- ✅ You can ask questions (CLI based chat)
- ✅ It remembers previous conversation context
- ✅ Every run, prompt, response, token usage appears in LangSmith
- ✅ You can ask recent / time-based questions (via a search tool)

<!-- User → LangGraph (state machine)
         ↓
   Conversation Memory
         ↓
      LLM (OpenAI / Groq)
         ↓
   Tool (Search for recent info)
         ↓
      LangSmith (logs everything) -->


<!-- User → LangGraph (state machine)
         ↓
   Conversation Memory
         ↓
      LLM (OpenAI / Groq)
         ↓
   Tool (Search for recent info)
         ↓
      LangSmith (logs everything) -->


<!-- Step 1: APIs -->
<!-- TODO: LLM Provider -->
<!-- GROQ_API_KEY =  your_groq_api_key_hereS3bW -->

<!-- LangSmith (mandatory for logs) -->
<!-- LANGCHAIN_API_KEY = your_langchain_api_key_here_5e433ffc2f -->

In [33]:
# STEP 2: Install required libraries
# !pip install langchain langgraph langsmith langchain-community langchain-groq tavily-python python-dotenv

In [34]:
from typing import List
from dotenv import load_dotenv
import os
import requests
import json
from langchain_core.messages import HumanMessage

# # 1️⃣ Load environment variables
load_dotenv()  # Load environment variables from .env file

# Euri API endpoint and token
EURI_API_URL = "https://api.euron.one/api/v1/euri/chat/completions"
EURI_API_TOKEN = os.getenv("EURI_API_TOKEN")

# Function to call Euri API with conversation history

def euri_chat(messages, model="gpt-4.1-nano", max_tokens=1000, temperature=0.7):
    payload = {
        "messages": messages,
        "model": model,
        "max_tokens": max_tokens,
        "temperature": temperature
    }
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {EURI_API_TOKEN}"
    }
    response = requests.post(EURI_API_URL, headers=headers, data=json.dumps(payload))
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error: {response.status_code}")
        print(response.text)
        return None

# `Define conversation state (memory)`

In [35]:
class ChatState(dict):
    """
    This state will travel through the graph.
    messages = full conversation history
    """
    messages:List

# 👉 This is why memory works
# LangGraph keeps passing this state forward.

# `Run interactive chat (YOU ask questions)`

In [36]:
if __name__ == "__main__":
    state = {
        "messages": []
    }
    print("LangGraph Chatbot (Euri API, type 'exit' to stop)\n")
    while True:
        user_input = input("You: ")
        if user_input.lower() == "exit":
            break
        # Add user message to conversation
        state["messages"].append({"role": "user", "content": user_input})
        # Call Euri API
        result = euri_chat(state["messages"])
        if result and "choices" in result and result["choices"]:
            # Euri API returns a list of choices, each with a message
            ai_message = result["choices"][0]["message"]
            # Euri API message may be a dict or a string
            if isinstance(ai_message, dict) and "content" in ai_message:
                # If content is a list (structured), join text parts
                content = ai_message["content"]
                if isinstance(content, list):
                    text = " ".join([c["text"] if isinstance(c, dict) and "text" in c else str(c) for c in content])
                else:
                    text = content
            else:
                text = str(ai_message)
            print("AI:", text)
            # Add assistant message to conversation
            state["messages"].append({"role": "assistant", "content": ai_message["content"] if isinstance(ai_message, dict) and "content" in ai_message else text})
        else:
            print("AI: (No response or error)")

LangGraph Chatbot (Euri API, type 'exit' to stop)

AI: India has many cities across its diverse regions. Some of the major cities include:

1. Delhi
2. Mumbai (Bombay)
3. Kolkata (Calcutta)
4. Chennai (Madras)
5. Bengaluru (Bangalore)
6. Hyderabad
7. Pune
8. Ahmedabad
9. Surat
10. Jaipur
11. Lucknow
12. Kanpur
13. Nagpur
14. Indore
15. Thane
16. Bhopal
17. Visakhapatnam
18. Pimpri-Chinchwad
19. Patna
20. Vadodara (Baroda)

These are just a few examples; India has hundreds of cities, towns, and municipalities, each with its own unique culture and significance.
AI: India has many cities across its diverse regions. Some of the major cities include:

1. Delhi
2. Mumbai (Bombay)
3. Kolkata (Calcutta)
4. Chennai (Madras)
5. Bengaluru (Bangalore)
6. Hyderabad
7. Pune
8. Ahmedabad
9. Surat
10. Jaipur
11. Lucknow
12. Kanpur
13. Nagpur
14. Indore
15. Thane
16. Bhopal
17. Visakhapatnam
18. Pimpri-Chinchwad
19. Patna
20. Vadodara (Baroda)

These are just a few examples; India has hundreds of citie